# 22. Consensus: GS vs SBI Model Assignment

Compares model assignments across four methods:
- GS-UM, GS-CP, SBI-UM, SBI-CP

Uses canonical consensus logic from `analysis.consensus`.
An animal is assigned when a majority of significant methods agree.

In [ ]:
%matplotlib inline
from shared_setup import *
from analysis.consensus import load_all_assignments, consensus_summary

In [ ]:
experiment, info = load_data()
assign_df = load_all_assignments(RESULTS_DIR, experiment)
print(consensus_summary(assign_df))

## 1. Full Assignment Table

In [ ]:
method_cols = ['GS-UM', 'GS-CP', 'SBI-UM', 'SBI-CP']
display_cols = ['id'] + method_cols + ['Consensus']
existing = [c for c in display_cols if c in assign_df.columns]
print(assign_df[existing].to_string(index=False))

## 2. P-Value Summary

How significant are the assignments?

In [ ]:
p_cols = [c for c in assign_df.columns if c.endswith('_p')]
if p_cols:
    p_display = ['id'] + p_cols
    print(assign_df[p_display].to_string(index=False, float_format='%.4f'))

## 3. Pairwise Agreement Matrix

What fraction of shared animals get the same assignment from each pair of methods?

In [ ]:
method_cols = ['GS-UM', 'GS-CP', 'SBI-UM', 'SBI-CP']
present = [c for c in method_cols if c in assign_df.columns]

if len(present) > 1:
    n = len(present)
    agree_matrix = np.zeros((n, n))

    for i, m1 in enumerate(present):
        for j, m2 in enumerate(present):
            valid = assign_df[m1].isin(['BE', 'SC']) & assign_df[m2].isin(['BE', 'SC'])
            if valid.sum() > 0:
                agree_matrix[i, j] = (
                    assign_df.loc[valid, m1] == assign_df.loc[valid, m2]
                ).mean()

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(agree_matrix, vmin=0, vmax=1, cmap='RdYlGn')
    ax.set_xticks(range(n))
    ax.set_xticklabels(present, rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(present)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{agree_matrix[i, j]:.0%}',
                    ha='center', va='center', fontsize=10)
    ax.set_title('Pairwise Method Agreement')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 methods to compare.')